# mini-vllm on an NVIDIA GPU

Run top to bottom. On Kaggle set **Settings → Accelerator → GPU T4 x2**;
on Colab, **Runtime → Change runtime type → T4 GPU**. Both free tiers give
Turing (sm_75), which is why this project hand-rolls its attention path rather
than depending on FlashAttention-2 — FA2 needs Ampere (sm_80) or newer.

Every cell prints paste-ready markdown. The point of the run is not only speed:
the slow test suite checks **logit parity against HuggingFace on CUDA**, which is
what makes the correctness claim hardware-independent.

Set `REPO` in the next cell before running.


In [ ]:
REPO = "https://github.com/Aayush121202/mini-vllm.git"  # <- set this

!git clone -q $REPO mini-vllm || echo 'already cloned'
%cd mini-vllm
!pip install -q -e ".[dev]"


## What hardware did we get?

Free-tier allocation varies. Record this next to any number you report — a
throughput figure without the device name is not a measurement.


In [ ]:
import torch

print('torch    ', torch.__version__)
print('cuda     ', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    print('device   ', torch.cuda.get_device_name(0))
    print('compute  ', f'sm_{major}{minor}')
    print('memory   ', f'{props.total_memory / 1024**3:.1f} GiB')
    print('bf16     ', torch.cuda.is_bf16_supported())


## 1. Fast suite

53 tests, no weights. Confirms the logic travels to a new machine.


In [ ]:
!python -m pytest -q -m 'not slow'


## 2. Slow suite — the actual prize

Downloads ~1GB of Qwen2.5-0.5B on first run, then checks on **CUDA**:

- prefill and cached-decode logits against HuggingFace
- 32 greedy tokens matching HF exactly
- batch invariance: 4 requests batched produce identical output to running each alone
- output survival across preempt-and-recompute
- prefix-cache reuse being indistinguishable from recomputing

Green here means correctness is validated on CPU, MPS **and** CUDA.


In [ ]:
!python -m pytest -q -m slow


## 3. Benchmarks

fp16 — T4 has no hardware bf16. Each prints a markdown table.

`--baseline hf` runs the same workload through HuggingFace `generate()`, held to
the same concurrency limit, so the comparison is against a real library rather
than against a worse version of this one.


In [ ]:
!python bench/benchmark.py offline --num-requests 64 --max-num-seqs 16 \
    --max-tokens 64 --dtype float16 --baseline hf


In [ ]:
!python bench/benchmark.py serving --num-requests 40 --rate 8 \
    --max-tokens 48 --max-num-seqs 16 --dtype float16 --trials 3


In [ ]:
!python bench/benchmark.py prefix --num-requests 8 --max-tokens 16 \
    --prefix-words 300 --dtype float16


## 4. Plot

Writes `bench/results/throughput-{light,dark}.png`. Download them if you want the
GPU curve in the README alongside the laptop one.


In [ ]:
!python -m bench.plot --dtype float16 --num-requests 64 --max-tokens 48


---

Paste the tables above into the README beside the Apple-M5 column, labelled with
the device name from the hardware cell. If a test fails or a benchmark regresses
on CUDA, that belongs in the README too — the project's claim is measurement
discipline, and a result that only holds on one machine is worth knowing about.
